In [ ]:
import sys

rawdataDir = r'C:\Users\pasillas-kim\Desktop\새 폴더\07001319_20250602_E125A_B1_T35_QC_4F_C101-200_TDKGSA2622\M03CH010[010]'
datFilename = r'07001319_20250602_E125A_B1_T35_QC_4F_C101-200_TDKGSA2622.dat'
cellID = '2622'

with open(rawdataDir+'\\'+datFilename, 'rb') as datFP:
    filecontent=datFP.read()        

In [37]:
new_filecontent=bytearray()

In [38]:
# StepEnd 추출
for i1 in range(int(len(filecontent)/104)):
    if (filecontent[i1*104+1] in range(1,5)):
        new_filecontent += filecontent[i1*104:(i1+1)*104]

In [39]:
import numpy as np
import pandas as pd

In [40]:

data_to_process = []

for step in range(int(len(new_filecontent)/104)):
    byteno = step*104
    if new_filecontent[byteno] in [8,11]:
        stype = 'Discharge'
    if new_filecontent[byteno] == 28:
        stype = 'Rest'
    if new_filecontent[byteno] in [2,4,5]:
        stype = 'Charge'
    data_to_process.append({\
        'StepNo': int.from_bytes(new_filecontent[byteno+6:byteno+8],sys.byteorder), \
        'StepType': stype, \
        'StepTime': int.from_bytes(new_filecontent[byteno+32:byteno+36],sys.byteorder)/100 if step==0 \
        else (int.from_bytes(new_filecontent[byteno+32:byteno+36],sys.byteorder)-int.from_bytes(new_filecontent[(byteno-104)+32:(byteno-104)+36],sys.byteorder))/100, \
        'TotalTime': int.from_bytes(new_filecontent[byteno+32:byteno+36],sys.byteorder)/100, \
        'Voltage': int.from_bytes(new_filecontent[byteno+16:byteno+20],sys.byteorder)/10000, \
        'Current': (pow(2,32) - int.from_bytes(new_filecontent[byteno+12:byteno+16],sys.byteorder))/10000 if stype == 'Discharge' \
        else int.from_bytes(new_filecontent[byteno+12:byteno+16],sys.byteorder)/10000, \
        'Capacity': int.from_bytes(new_filecontent[byteno+20:byteno+24],sys.byteorder)/1000, \
        'Power': (pow(2,32) - int.from_bytes(new_filecontent[byteno+24:byteno+28],sys.byteorder))/1000 if stype == 'Discharge' \
        else int.from_bytes(new_filecontent[byteno+24:byteno+28],sys.byteorder)/1000, \
        'WattHour': int.from_bytes(new_filecontent[byteno+28:byteno+32],sys.byteorder)/1000, \
        'Impedance': int.from_bytes(new_filecontent[byteno+44:byteno+48],sys.byteorder)/1000, \
        'Temp': int.from_bytes(new_filecontent[byteno+8:byteno+10],sys.byteorder)/10, \
        'Cy': new_filecontent[byteno+40], \
        'TotalCycle': int.from_bytes(new_filecontent[byteno+4:byteno+6],sys.byteorder), \
        'AvgVoltage': int.from_bytes(new_filecontent[byteno+72:byteno+76],sys.byteorder)/10000, \
        'AvgCurrent': (pow(2,32) - int.from_bytes(new_filecontent[byteno+76:byteno+80],sys.byteorder))/10000 if stype=='Discharge' \
        else int.from_bytes(new_filecontent[byteno+76:byteno+80],sys.byteorder)/10000, \
        'ChargeCapacity': int.from_bytes(new_filecontent[byteno+52:byteno+56],sys.byteorder)/1000, \
        'DischargeCapacity': int.from_bytes(new_filecontent[byteno+56:byteno+60],sys.byteorder)/1000})


In [41]:
df = pd.DataFrame(data_to_process)
df

,StepNo,StepType,StepTime,TotalTime,Voltage,Current,Capacity,Power,WattHour,Impedance,Temp,Cy,TotalCycle,AvgVoltage,AvgCurrent,ChargeCapacity,DischargeCapacity
0,2,Rest,10800.0,10800.0,3.6312,0.0000,0.000,0.000,0.000,0.000,35.2,1,1,3.6311,0.0000,0.000,0.000
1,5,Charge,8354.6,19154.6,4.3500,6.2747,91.470,27.294,364.525,0.881,35.6,2,2,4.0068,39.4055,91.470,0.000
2,6,Rest,1800.0,20954.6,4.3281,0.0000,0.000,0.000,0.000,0.000,35.3,2,2,4.3322,0.0000,0.000,0.000
3,7,Discharge,11286.4,32241.0,2.5000,41.8414,131.169,104.603,488.307,0.994,36.1,2,2,3.7226,41.8433,0.000,131.169
4,8,Rest,1800.0,34041.0,3.0271,0.0000,0.000,0.000,0.000,0.000,35.5,2,2,2.9946,0.0000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
783,19,Charge,117.4,1109628.7,4.1580,102.4169,3.340,425.849,13.837,0.000,40.4,3,58,4.1421,102.4222,3.340,0.000
784,20,Rest,10.0,1109638.7,4.0706,0.0000,0.000,0.000,0.000,0.000,40.4,3,58,4.0773,0.0000,0.000,0.000
785,21,Charge,3115.5,1112754.2,4.2849,6.2790,28.375,26.904,119.446,0.668,35.9,3,58,4.2258,32.7817,28.375,0.000
786,22,Rest,1800.0,1114554.2,4.2672,0.0000,0.000,0.000,0.000,0.000,35.4,3,58,4.2702,0.0000,0.000,0.000


In [42]:
stepno_charge_normal = []
stepno_charge_fast = [i for i in range(11,22)]

stepno_charge = stepno_charge_normal + stepno_charge_fast

stepno_discharge = [23]

stepno_rest_after_charge = [22]

stepno_rest_after_discharge = [24]
stepno_rest_before_charge = [8] + stepno_rest_after_discharge

chgOCV = df.loc[df['StepNo'].isin(stepno_rest_after_charge), ['Voltage']].to_numpy()
dchOCV = df.loc[df['StepNo'].isin(stepno_rest_after_discharge), ['Voltage']].to_numpy()
df2 = df.loc[df['StepNo'].isin(stepno_charge)]
chgCapa = pd.pivot_table(df2, values="ChargeCapacity", index="TotalCycle", aggfunc="sum").to_numpy()
dchCapa = df.loc[df['StepNo'].isin(stepno_discharge), ['DischargeCapacity']].to_numpy()
chgBeforeTemp = df.loc[df['StepNo'].isin(stepno_rest_before_charge), ['Temp']].to_numpy()
chgTemp = df.loc[df['StepNo'].isin(stepno_charge_normal + [stepno_charge_fast[-1]]), ['Temp']].to_numpy()
dchBeforeTemp = df.loc[df['StepNo'].isin(stepno_rest_after_charge), ['Temp']].to_numpy()
dchTemp = df.loc[df['StepNo'].isin(stepno_discharge), ['Temp']].to_numpy()
chgE = pd.pivot_table(df2, values="WattHour", index="TotalCycle", aggfunc="sum").to_numpy()
dchE = df.loc[df['StepNo'].isin(stepno_discharge), ['WattHour']].to_numpy()

std_sz = min(chgOCV.shape[0],dchOCV.shape[0],chgCapa.shape[0],dchCapa.shape[0],chgBeforeTemp.shape[0],chgTemp.shape[0],dchBeforeTemp.shape[0],dchTemp.shape[0],chgE.shape[0],dchE.shape[0])

df_out = pd.DataFrame(np.concatenate((chgOCV[:std_sz],dchOCV[:std_sz],chgCapa[:std_sz],dchCapa[:std_sz],chgBeforeTemp[:std_sz],chgTemp[:std_sz],dchBeforeTemp[:std_sz],dchTemp[:std_sz],chgE[:std_sz],dchE[:std_sz]),axis=1))
df_out.to_csv(cellID + "_output.csv",index=False,header=False)

In [8]:
stepno = 4
byteno = (stepno-1)*104

# print(new_filecontent[byteno])
if new_filecontent[byteno] in [8,11]:
    print("유형 : 방전")
    print("전류 : ", (pow(2,32) - int.from_bytes(new_filecontent[byteno+12:byteno+16],sys.byteorder))/10000)
    print("출력 : ", (pow(2,32) - int.from_bytes(new_filecontent[byteno+24:byteno+28],sys.byteorder))/1000)
    print("평균전류 : ", (pow(2,32) - int.from_bytes(new_filecontent[byteno+76:byteno+80],sys.byteorder))/10000)
if new_filecontent[byteno] == 28:
    print("유형 : Rest")
    print("전류 : 0")
if new_filecontent[byteno] in [2,4,5]:
    print("유형 : 충전")    
    print("전류 : ", int.from_bytes(new_filecontent[byteno+12:byteno+16],sys.byteorder)/10000)
    print("출력 : ", int.from_bytes(new_filecontent[byteno+24:byteno+28],sys.byteorder)/1000)
    print("평균전류 : ", int.from_bytes(new_filecontent[byteno+76:byteno+80],sys.byteorder)/10000)
    
print("총 사이클 : ", int.from_bytes(new_filecontent[byteno+4:byteno+6],sys.byteorder))
print("Step No : ", int.from_bytes(new_filecontent[byteno+6:byteno+8],sys.byteorder))
print("Voltage(V) : ",int.from_bytes(new_filecontent[byteno+16:byteno+20],sys.byteorder)/10000) #Voltage
print("Temp.(oC) : ",new_filecontent[byteno+8]/10+25.6)
print("용량(Ah) : ", int.from_bytes(new_filecontent[byteno+20:byteno+24],sys.byteorder)/1000)
print("WattHour(Wh) : ", int.from_bytes(new_filecontent[byteno+28:byteno+32],sys.byteorder)/1000)
print("누적시간(s) : ", int.from_bytes(new_filecontent[byteno+32:byteno+36],sys.byteorder)/100)
print("Index : ", int.from_bytes(new_filecontent[byteno+36:byteno+40],sys.byteorder))
print("Cy : ", new_filecontent[byteno+40])
print("DCIR (mOhm) : ", int.from_bytes(new_filecontent[byteno+44:byteno+48],sys.byteorder)/1000)
if new_filecontent[byteno+51]!=255:
    print("총용량 (Ah) : ", int.from_bytes(new_filecontent[byteno+48:byteno+52],sys.byteorder)/1000)
else:
    print("총용량 (Ah) : ", -(pow(2,32) - int.from_bytes(new_filecontent[byteno+48:byteno+52],sys.byteorder))/1000)

print("충전용량 (Ah) : ", int.from_bytes(new_filecontent[byteno+52:byteno+56],sys.byteorder)/1000)
print("방전용량 (Ah) : ", int.from_bytes(new_filecontent[byteno+56:byteno+60],sys.byteorder)/1000)
if new_filecontent[byteno+63]!=255:
    print("총WattHour (Wh) : ", int.from_bytes(new_filecontent[byteno+60:byteno+64],sys.byteorder)/1000)
else:
    print("총WattHour (Wh) : ", -(pow(2,32) - int.from_bytes(new_filecontent[byteno+60:byteno+64],sys.byteorder))/1000)

print("충전WattHour (Wh) : ", int.from_bytes(new_filecontent[byteno+64:byteno+68],sys.byteorder)/1000)
print("방전WattHour (Wh) : ", int.from_bytes(new_filecontent[byteno+68:byteno+72],sys.byteorder)/1000)
print("평균전압 (V) : ", int.from_bytes(new_filecontent[byteno+72:byteno+76],sys.byteorder)/10000)
print("Tamb : ", new_filecontent[byteno+82]/10+25.6)
print("CV시간 (s) : ", int.from_bytes(new_filecontent[byteno+92:byteno+96],sys.byteorder)/1000)

print("CV용량 (Ah) : ", int.from_bytes(new_filecontent[byteno+96:byteno+98],sys.byteorder)/1000)

유형 : 방전
전류 :  41.8673
출력 :  104.664
평균전류 :  41.8677
총 사이클 :  2
Step No :  7
Voltage(V) :  2.4999
Temp.(oC) :  36.1
용량(Ah) :  132.705
WattHour(Wh) :  493.931
누적시간(s) :  35930.5
Index :  35934
Cy :  2
DCIR (mOhm) :  0.979
총용량 (Ah) :  -0.13
충전용량 (Ah) :  0.0
방전용량 (Ah) :  132.705
총WattHour (Wh) :  18.291
충전WattHour (Wh) :  0.0
방전WattHour (Wh) :  493.931
평균전압 (V) :  3.7222
Tamb :  34.8
CV시간 (s) :  0.0
CV용량 (Ah) :  0.0


In [ ]:
# StepEnd 추출
for i1 in range(int(len(filecontent)/104)):
    if filecontent[i1*104+1] != 0:
        import sys
        print(pow(2,32)-int.from_bytes(filecontent[(35931*104+12):(35931*104+16)],sys.byteorder))

In [53]:
raw_index = 88
# filecontent[(raw_index-1)*80 + position -1]

for i in range(13):
    for i2 in range(8):
        print(str(new_filecontent[(raw_index-1)*104 + i*8+i2]) + '\t', end='')
    print('\n')


8	2	0	113	8	0	23	0	

5	1	101	119	148	157	249	255	

163	97	0	0	98	232	1	0	

122	103	254	255	173	17	7	0	

6	209	230	0	97	79	2	0	

3	0	0	0	44	4	0	0	

1	0	0	0	0	0	0	0	

98	232	1	0	155	141	0	0	

0	0	0	0	173	17	7	0	

191	144	0	0	137	157	249	255	

45	0	248	0	250	0	250	0	

0	243	200	0	0	0	0	0	

0	0	0	0	225	144	2	0	



In [46]:
new_filecontent[87*104+1]

10

In [ ]:
import requests
from bs4 import BeautifulSoup
import urllib3
import re

# SSL 경고 무시
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def namu_surf():
    # 실제 브라우저처럼 보이게 만드는 헤더 (중요)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    }

    while True:
        keyword = input("\n[나무위키 검색 (종료: q)]: ")
        if keyword.lower() == 'q': break
        
        url = f"https://namu.wiki/w/{keyword}"
        
        try:
            # verify=False로 회사 보안망 통과
            res = requests.get(url, headers=headers, verify=False, timeout=10)
            
            if res.status_code == 403:
                print("🚨 접근 거부 (보안망에서 막혔을 수 있습니다.)")
                continue
                
            soup = BeautifulSoup(res.text, 'html.parser')
            
            # 나무위키 본문 영역 추출 (클래스 구조에 따라 최적화)
            # 보통 'wiki-heading-content' 또는 특정 div 안에 본문이 들어있습니다.
            content = soup.find_all(['p', 'h1', 'h2', 'h3'])
            
            print("\n" + "="*60)
            print(f"📖 문서 제목: {keyword}")
            print("="*60 + "\n")
            
            for p in content:
                text = p.get_text().strip()
                if text:
                    # 너무 긴 공백이나 특수문자 정제
                    clean_text = re.sub(r'\[\d+\]', '', text) # 각주 제거 ([1], [2] 등)
                    print(clean_text)
                    print() # 가독성을 위한 한 줄 띄우기
                    
            print("="*60)
            print("--- 문서 끝 ---")

        except Exception as e:
            print(f"❌ 오류가 발생했습니다: {e}")

if __name__ == "__main__":
    namu_surf()